# 1975 Genetic Algorithms

John Holland's 1975 book *Adaptation in Natural and Artificial Systems* helped establish genetic algorithms as a way to search for good solutions by imitating evolution. Instead of solving a problem with hand-written rules, a genetic algorithm keeps a population of candidate answers and improves them through selection, crossover, and mutation.

## High-Level Ideas

A genetic algorithm searches by repeatedly improving a population of candidate solutions:

- **Chromosomes** encode candidate solutions as simple pieces, such as bits, numbers, or symbols.
- **Fitness** scores how well each chromosome solves the problem.
- **Selection** gives stronger candidates more chances to become parents.
- **Crossover** mixes pieces from two parents to create a child.
- **Mutation** occasionally changes a gene so the search can explore new possibilities.
- **Generations** repeat this cycle until the population finds a good enough answer.

The idea is not that evolution guarantees the perfect answer. The useful part is that a population can explore many possible answers while gradually concentrating effort around the most promising ones.

## Example: Packing a Field Kit

A classic small genetic algorithm example is the 0/1 knapsack problem: choose the most valuable set of items without exceeding a weight limit. Here, each chromosome is a string of bits. A `1` means the item is packed, and a `0` means it is left behind.

This version imagines packing a field kit with limited carrying capacity. The genetic algorithm searches for a high-value pack using selection, crossover, and mutation. Because the item list is intentionally small, the notebook also checks the result against an exact brute-force answer at the end.

In [1]:
import itertools
import random


items = [
    {"name": "water filter", "weight": 3, "value": 10},
    {"name": "first aid kit", "weight": 4, "value": 9},
    {"name": "radio", "weight": 5, "value": 8},
    {"name": "food pack", "weight": 6, "value": 12},
    {"name": "flashlight", "weight": 2, "value": 6},
    {"name": "battery pack", "weight": 3, "value": 7},
    {"name": "blanket", "weight": 4, "value": 6},
    {"name": "rope", "weight": 5, "value": 7},
    {"name": "map", "weight": 1, "value": 4},
    {"name": "jacket", "weight": 4, "value": 8},
]

capacity = 15


class Individual:
    def __init__(self, chromosome, fitness):
        self.chromosome = chromosome
        self.fitness = fitness


class KnapsackGeneticAlgorithm:
    def __init__(self, items, capacity, population_size=80, mutation_rate=0.04, seed=1975):
        self.items = items
        self.capacity = capacity
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        self.random_generator = random.Random(seed)

    def run(self, max_generations=100, target_fitness=None):
        population = self.create_initial_population()
        history = []

        for generation in range(max_generations + 1):
            best_individual = self.best_individual(population)
            history.append((generation, best_individual))

            if target_fitness is not None and best_individual.fitness >= target_fitness:
                break

            if generation == max_generations:
                break

            population = self.next_generation(population)

        return history[-1][1], history

    def create_initial_population(self):
        return [self.make_individual(self.random_chromosome()) for _ in range(self.population_size)]

    def random_chromosome(self):
        return tuple(
            self.random_generator.choice((0, 1))
            for _ in self.items
        )

    def make_individual(self, chromosome):
        return Individual(
            chromosome=tuple(chromosome),
            fitness=self.score_fitness(chromosome),
        )

    def score_fitness(self, chromosome):
        total_weight, total_value = self.solution_totals(chromosome)

        if total_weight > self.capacity:
            return 0

        return total_value

    def selected_items(self, chromosome):
        return [
            item
            for gene, item in zip(chromosome, self.items)
            if gene == 1
        ]

    def solution_totals(self, chromosome):
        selected_items = self.selected_items(chromosome)
        total_weight = sum(item["weight"] for item in selected_items)
        total_value = sum(item["value"] for item in selected_items)

        return total_weight, total_value

    def best_individual(self, population):
        return max(population, key=lambda individual: individual.fitness)

    def next_generation(self, population):
        next_population = [self.best_individual(population)]

        while len(next_population) < self.population_size:
            parent_a = self.select_parent(population)
            parent_b = self.select_parent(population)
            child_chromosome = self.crossover(parent_a.chromosome, parent_b.chromosome)
            child_chromosome = self.mutate(child_chromosome)
            next_population.append(self.make_individual(child_chromosome))

        return next_population

    def select_parent(self, population, tournament_size=3):
        tournament = self.random_generator.sample(population, tournament_size)
        return self.best_individual(tournament)

    def crossover(self, chromosome_a, chromosome_b):
        split_index = self.random_generator.randint(1, len(chromosome_a) - 1)
        return chromosome_a[:split_index] + chromosome_b[split_index:]

    def mutate(self, chromosome):
        return tuple(
            1 - gene if self.random_generator.random() < self.mutation_rate else gene
            for gene in chromosome
        )


def exact_best_individual(items, capacity):
    scorer = KnapsackGeneticAlgorithm(items, capacity, population_size=1)
    best_individual = None

    for chromosome in itertools.product((0, 1), repeat=len(items)):
        individual = scorer.make_individual(chromosome)

        if best_individual is None or individual.fitness > best_individual.fitness:
            best_individual = individual

    return best_individual


def chromosome_to_text(chromosome):
    return "".join(str(gene) for gene in chromosome)


def print_solution(label, individual, solver):
    total_weight, total_value = solver.solution_totals(individual.chromosome)
    selected_names = [item["name"] for item in solver.selected_items(individual.chromosome)]

    print(label)
    print(f"  chromosome: {chromosome_to_text(individual.chromosome)}")
    print(f"  items: {', '.join(selected_names)}")
    print(f"  weight: {total_weight}/{solver.capacity}")
    print(f"  value: {total_value}")

In [2]:
exact_best = exact_best_individual(items, capacity)

genetic_algorithm = KnapsackGeneticAlgorithm(
    items=items,
    capacity=capacity,
    population_size=80,
    mutation_rate=0.04,
    seed=1975,
)

best_individual, history = genetic_algorithm.run(
    max_generations=100,
    target_fitness=exact_best.fitness,
)

print("progress:")

for generation, individual in history[::10]:
    total_weight, total_value = genetic_algorithm.solution_totals(individual.chromosome)
    print(
        f"generation {generation:>3}: "
        f"chromosome={chromosome_to_text(individual.chromosome)} "
        f"weight={total_weight:>2}/{capacity} "
        f"value={total_value:>2} "
        f"fitness={individual.fitness:>2}"
    )

final_generation, final_individual = history[-1]
if final_generation % 10 != 0:
    total_weight, total_value = genetic_algorithm.solution_totals(final_individual.chromosome)
    print(
        f"generation {final_generation:>3}: "
        f"chromosome={chromosome_to_text(final_individual.chromosome)} "
        f"weight={total_weight:>2}/{capacity} "
        f"value={total_value:>2} "
        f"fitness={final_individual.fitness:>2}"
    )

print()
print_solution("genetic algorithm solution:", best_individual, genetic_algorithm)
print()
print_solution("exact solution for this small item list:", exact_best, genetic_algorithm)

assert best_individual.fitness == exact_best.fitness

print("\nGenetic algorithm matched the exact best value for this small kit.")

progress:
generation   0: chromosome=1000110011 weight=13/15 value=35 fitness=35
generation   5: chromosome=1001110010 weight=15/15 value=39 fitness=39

genetic algorithm solution:
  chromosome: 1001110010
  items: water filter, food pack, flashlight, battery pack, map
  weight: 15/15
  value: 39

exact solution for this small item list:
  chromosome: 1001110010
  items: water filter, food pack, flashlight, battery pack, map
  weight: 15/15
  value: 39

Genetic algorithm matched the exact best value for this small kit.


## Why This Mattered

Genetic algorithms gave AI researchers a general-purpose way to search large, messy spaces where a direct formula was hard to write. They became influential in optimization, scheduling, design, machine learning experiments, and artificial life because they turned adaptation itself into a computational method.